In [107]:
import sqlite3
import pandas as pd
import datetime 
conn = sqlite3.connect('../data/checking-logs.sqlite')


In [108]:
avg_first_view_query = """
SELECT AVG(julianday(first_view_ts)) 
FROM test 
WHERE first_view_ts IS NOT NULL
"""
avg_first_view = pd.io.sql.read_sql(avg_first_view_query, conn).iloc[0,0]
print((avg_first_view))

2458966.5278444653


In [109]:
# Используя только один запрос для каждой из групп, создайте два фрейма данных: test_results и control_results со столбцами time и avg_diff и только двумя строками
# время должно иметь значения: после и до
# avg_diff содержит среднее значение дельты для всех пользователей за период времени до их первого посещения страницы и после него
# учитывайте только тех пользователей, у которых есть наблюдения до и после


test_query = """
WITH user_stats AS (
    SELECT 
        t.uid,
        AVG(CASE 
            WHEN t.first_commit_ts < t.first_view_ts 
            THEN (julianday(datetime(d.deadlines, 'unixepoch')) - julianday(t.first_commit_ts)) * 24 
            ELSE NULL 
        END) AS before_diff,
        AVG(CASE 
            WHEN t.first_commit_ts >= t.first_view_ts 
            THEN (julianday(datetime(d.deadlines, 'unixepoch')) - julianday(t.first_commit_ts)) * 24 
            ELSE NULL 
        END) AS after_diff
    FROM 
        test t
    JOIN 
        deadlines d ON t.labname = d.labs
    WHERE 
        t.labname != 'project1'
        AND t.first_view_ts IS NOT NULL
    GROUP BY 
        t.uid
    HAVING
        before_diff IS NOT NULL
        AND after_diff IS NOT NULL
)
SELECT 
    'before' AS time,
    AVG(before_diff) AS avg_diff
FROM 
    user_stats
UNION ALL
SELECT 
    'after' AS time,
    AVG(after_diff) AS avg_diff
FROM 
    user_stats
"""
test_results = pd.io.sql.read_sql(test_query, conn)
print(test_results)

     time    avg_diff
0  before   66.679398
1   after  100.178032


In [110]:
control_query = f"""
WITH user_stats AS (
    SELECT 
        c.uid,
        AVG(CASE 
            WHEN julianday(c.first_commit_ts) < {avg_first_view} 
            THEN (julianday(datetime(d.deadlines, 'unixepoch')) - julianday(c.first_commit_ts)) * 24 
            ELSE NULL 
        END) AS before_diff,
        AVG(CASE 
            WHEN julianday(c.first_commit_ts) >= {avg_first_view} 
            THEN (julianday(datetime(d.deadlines, 'unixepoch')) - julianday(c.first_commit_ts)) * 24 
            ELSE NULL 
        END) AS after_diff
    FROM 
        control c
    JOIN 
        deadlines d ON c.labname = d.labs
    WHERE 
        c.labname != 'project1'
    GROUP BY 
        c.uid
    HAVING
        before_diff IS NOT NULL
        AND after_diff IS NOT NULL
)
SELECT 
    'before' AS time,
    AVG(before_diff) AS avg_diff
FROM 
    user_stats
UNION ALL
SELECT 
    'after' AS time,
    AVG(after_diff) AS avg_diff
FROM 
    user_stats
"""
control_results = pd.io.sql.read_sql(control_query, conn)
print(control_results)


     time   avg_diff
0  before  98.467698
1   after  99.803422


In [111]:
conn.close()

In [112]:
test_before = test_results[test_results['time'] == 'before']['avg_diff'].values[0]
test_after = test_results[test_results['time'] == 'after']['avg_diff'].values[0]
control_before = control_results[control_results['time'] == 'before']['avg_diff'].values[0]
control_after = control_results[control_results['time'] == 'after']['avg_diff'].values[0]

if (test_after < test_before) and (abs(control_after - control_before) < abs(test_after - test_before)/2):
    print("\nВывод: Гипотеза подтвердилась - новостная лента повлияла на поведение студентов")
    print("Студенты начали выполнять задания раньше после просмотра ленты")
else:
    print("\nВывод: Гипотеза не подтвердилась - новостная лента не оказала значимого влияния")



Вывод: Гипотеза не подтвердилась - новостная лента не оказала значимого влияния
